# TUTORIAL 06: KIRBY CALCULUS AND CHARACTERISTIC CLASSES

In the previous tutorials, we treated 4-manifolds as abstract algebraic matrices (Intersection Forms). But topology is deeply geometric. 

How do we actually "build" a 4-manifold in space? We use **Kirby Calculus**.

In this tutorial, we will learn:
1. **Kirby Diagrams**: How to represent 4-manifolds as knotted, linked circles in 3-dimensional space.
2. **Handle Slides & Blow-ups**: The geometric moves that change the diagram but preserve the manifold's topology.
3. **Characteristic Classes**: How to use Wu's Formula to extract the 2nd Stiefel-Whitney class ($w_2$) and mathematically prove if a manifold admits a **Spin structure**.
4. **The Hirzebruch Signature Theorem**: How to compute Pontryagin classes ($p_1$) and verify the signature formula.

In [ ]:
import numpy as np
from pysurgery import KirbyDiagram, check_spin_structure, extract_pontryagin_p1, verify_hirzebruch_signature
print("Kirby Calculus and Characteristic Classes engine loaded.")

## 1. Kirby Diagrams: Drawing 4D in 3D

A 4-manifold can be built by gluing "handles" together (0-handles, 1-handles, 2-handles, etc.). 
The most important part of a simply-connected 4-manifold is its 2-handles (which correspond to the 2-dimensional holes in $H_2$).

A **2-handle** ($D^2 \times D^2$) is glued into a 4-manifold along its boundary ($S^1 \times D^2$, a solid torus). 
This means we can specify a 4-manifold completely by drawing a collection of knotted circles (the $S^1$ cores) inside a 3-sphere $S^3$, and giving each circle an integer **framing** (how many times the $D^2$ twists as it goes around the circle).

### The Linking Matrix
If we have a Kirby diagram of knots $K_1, K_2, \dots$, we form a symmetric **Linking Matrix**:
- The **diagonal elements** are the framings of the individual knots.
- The **off-diagonal elements** are the linking numbers $lk(K_i, K_j)$ (how many times knot $i$ passes through knot $j$).

Miraculously, **the linking matrix of the Kirby diagram is exactly the Intersection Form of the 4-manifold!**

In [ ]:
# Let's create a Kirby Diagram from a linking matrix.
# Consider two unknots that are linked once (lk = 1), both with framing 0.
# This represents S^2 x S^2.

linking_matrix = np.array([
    [0, 1],
    [1, 0]
])

diagram = KirbyDiagram.from_matrix(linking_matrix)
print("Initial Kirby Diagram created.")
print(f"Framings (knots self-twists): {diagram.framings}")
print(f"Linking Matrix:\n{diagram.linking_matrix}")

## 2. Geometric Surgery: Kirby Moves

We can change the Kirby diagram without changing the underlying 4-manifold (or changing it in a strictly controlled way) using **Kirby Moves**.

### Move 1: The Handle Slide
You can slide one 2-handle over another. If we slide knot $K_s$ over knot $K_t$, we geometrically push $K_s$ along a path until it merges with $K_t$.

Algebraically, `pysurgery` executes a highly specific change-of-basis matrix multiplication to recalculate the linking numbers. The framing of the source knot changes dramatically:
$$ f_s \to f_s + f_t + 2 \cdot lk(K_s, K_t) $$

In [ ]:
print("--- Performing a Handle Slide ---")
# We slide Knot 0 over Knot 1.
# Because f_0=0, f_1=0, and lk(0,1)=1, the new framing of Knot 0 should be:
# 0 + 0 + 2(1) = 2.

slided_diagram = diagram.handle_slide(source_idx=0, target_idx=1)
print(f"New Linking Matrix:\n{slided_diagram.linking_matrix}")

print("\nNotice the new matrix is [[2, 1], [1, 0]].")
print("Even though the matrix changed, the manifold is still strictly homeomorphic to S^2 x S^2!")

### Move 2: The Blow-Up
A blow-up adds an isolated unknot with a framing of $\pm 1$. 
Geometrically, this is not an equivalence. It literally performs a **connected sum** with the Complex Projective Plane $\mathbb{CP}^2$ (if $+1$) or $\overline{\mathbb{CP}}^2$ (if $-1$).

In [ ]:
print("\n--- Performing a Blow-Up ---")
# We blow up the manifold with a -1 unknot.
blown_up_diagram = slided_diagram.blow_up(sign=-1)

print(f"Matrix after Blow-Up:\n{blown_up_diagram.linking_matrix}")
print("\nThe manifold is now (S^2 x S^2) # (-CP^2).")

## 3. Characteristic Classes and Spin Structures

In differential geometry, a manifold is **Spin** if its tangent bundle doesn't have a certain "twist" (it allows the existence of spinors, crucial for quantum physics).

Topologically, a manifold admits a Spin structure if and only if its **2nd Stiefel-Whitney Class** $w_2 \in H^2(M; \mathbb{Z}_2)$ is exactly zero.

How do we find $w_2$? We use **Wu's Formula**. 
For a 4-manifold, Wu's formula states that $w_2$ is the unique characteristic element of the Intersection Form modulo 2. Mathematically, for every vector $x$:
$$ Q(x, w_2) \equiv Q(x, x) \pmod 2 $$

`pysurgery` rigorously evaluates this by inverting the intersection matrix over the finite field $GF(2)$ using SymPy.

In [ ]:
print("--- Evaluating Spin Structures ---")

# 1. Original Diagram (S^2 x S^2)
q_original = diagram.extract_intersection_form()
spin_report_1 = check_spin_structure(q_original)
print(f"Original S^2 x S^2:\n{spin_report_1}\n")

# 2. Blown-Up Diagram ((S^2 x S^2) # (-CP^2))
q_blown_up = blown_up_diagram.extract_intersection_form()
spin_report_2 = check_spin_structure(q_blown_up)
print(f"After Blow-Up (-CP^2):\n{spin_report_2}")

### Mathematical Insight
Look at the results above! 
$S^2 \times S^2$ is a Spin manifold ($w_2 = 0$). 
But as soon as we blew it up by adding a $-1$ framed knot, the manifold became **Non-Spin**! The Stiefel-Whitney class $w_2$ jumped to `e_2` (the basis vector of the new knot).

This perfectly aligns with the fact that $\mathbb{CP}^2$ and $\overline{\mathbb{CP}}^2$ are intrinsically non-Spin. `pysurgery` can track these profound geometric twists purely from the algebraic linking matrix!

## 4. The Hirzebruch Signature Theorem & Pontryagin Classes

Beyond mod 2 arithmetic (Stiefel-Whitney classes), we have integer characteristic classes representing the true geometry of the tangent bundle: the **Pontryagin classes** $p_i \in H^{4i}(M; \mathbb{Z})$.

The holy grail of 4-manifold topology is the **Hirzebruch Signature Theorem**, which inextricably links the purely algebraic signature of the intersection form to the geometric integration of the first Pontryagin class over the fundamental class $[M]$:
$$ \sigma(M) = \frac{1}{3} \langle p_1(M), [M] \rangle $$

`pysurgery` leverages this to computationally extract $p_1(M)[M]$ and strictly verify if the geometry matches the algebra.

In [ ]:
print("--- Verifying the Hirzebruch Signature Theorem ---")

# Let's analyze our blown-up manifold: (S^2 x S^2) # (-CP^2)
sig = q_blown_up.signature()
print(f"Algebraic Signature of Q: {sig}")

# Evaluate p_1(M)[M]
p1_eval = extract_pontryagin_p1(q_blown_up)
print(f"Geometric Evaluation p_1(M)[M]: {p1_eval}")

# Verify the Theorem
is_valid = verify_hirzebruch_signature(q_blown_up, p1_eval)
print(f"Hirzebruch Theorem Holds? {is_valid}")
print("Explanation: p_1(M)[M] / 3 = -3 / 3 = -1, which perfectly matches the signature!")

## Conclusion
In this tutorial, we learned how to:
1. Transition from pure algebraic matrices to geometric **Kirby Diagrams**.
2. Perform **Handle Slides** and **Blow-ups** to geometrically alter the manifold.
3. Use profound characteristic class theorems (Wu's Formula) to rigorously detect **Spin structures** ($w_2$).
4. Compute the first **Pontryagin Class** ($p_1$) and verify the crown jewel of 4-manifolds: the **Hirzebruch Signature Theorem**.